# Streaming a remote ENCODE CNDB file with CNDBTools

This optional notebook demonstrates the `cndb-stream` backend for OpenMiChroM CNDBTools. It reads the embedded index and small coordinate byte ranges from a remote ENCODE CNDB file. It does **not** download the full 139 GB CNDB file.

Requirements:

```bash
pip install cndb-stream
```

This notebook requires internet access and is not intended for normal CI.

In [ ]:
import numpy as np

from OpenMiChroM.CndbTools import CndbTools

In [ ]:
ENCODE_URL = "https://encode-public.s3.amazonaws.com/2023/02/02/7f75d816-342a-4b49-adbd-aaa499dc5201/ENCFF161DID.cndb"
TRAJECTORY = "replica1_chr1"
FRAMES = [1, 10, 100, 1000]
BEAD_SELECTION = range(0, 100)

In [ ]:
tools = CndbTools.from_remote(
    h5_url=ENCODE_URL,
    trajectory=TRAJECTORY,
    index_cache_path="/tmp/ENCFF161DID.embedded-index.json.gz",
)

print("trajectory:", tools.current_trajectory)
print("n_frames:", tools.Nframes)
print("n_beads:", tools.Nbeads)
print("available trajectories:", len(tools.trajectories))

In [ ]:
xyz = tools.xyz(frames=FRAMES, beadSelection=BEAD_SELECTION)
expected_data_bytes = len(FRAMES) * len(BEAD_SELECTION) * 3 * np.dtype("float32").itemsize

print("xyz shape:", xyz.shape)
print("expected coordinate bytes:", expected_data_bytes)
print("stream stats:", tools.stream_stats())

In [ ]:
rg = tools.compute_RG(xyz)
print("radius of gyration per selected frame:")
for frame, value in zip(FRAMES, rg):
    print(frame, value)

In [ ]:
subset = xyz[0, :25]
diff = subset[:, None, :] - subset[None, :, :]
distance_matrix = np.sqrt(np.sum(diff * diff, axis=-1))
print("distance matrix shape:", distance_matrix.shape)
print(distance_matrix[:5, :5])